In [1]:
import os
import sys
sys.path.append("..")
import json
from torchinfo import summary

from models.xLSTM_model import xLSTMRegressor, xLSTMRegressor_v2
from models.LSTM_model import LSTMRegressor

# Comparison Baseline

In [3]:
task = "comparison_baseline_cv"
interval = 5
config = "default"
seq_size = interval * 100
context_time_mins = 5
seq_len = context_time_mins * 60 * 100 // seq_size
with open(f"../config/{task}/lstm_{config}_{interval}sec_config.json", "r") as file:
    config_lstm = json.load(file)
with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
    config_xlstm = json.load(file)
# config = "v1"
# with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
#     config_xlstm2 = json.load(file)

lstm_model = LSTMRegressor(**config_lstm)
# xlstm_model = xLSTMRegressor(**config_xlstm)
xlstm_model_v2 = xLSTMRegressor_v2(**config_xlstm)
# xlstm_model_v3 = xLSTMRegressor_v3(**config_xlstm)

# xlstm_model2 = xLSTMRegressor(**config_xlstm2)

In [4]:
summary(model=lstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                  Input Shape     Output Shape    Param #         Param %         Trainable
LSTMRegressor (LSTMRegressor)            [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                     [128, 60, 500]  [128, 60, 96]   48,096           39.20%         True
├─LSTM (lstm)                            [128, 60, 96]   [128, 60, 96]   74,496           60.72%         True
├─Linear (fc)                            [128, 96]       [128, 1]        97                0.08%         True
Total params: 122,689
Trainable params: 122,689
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 578.30
Input size (MB): 15.36
Forward/backward pass size (MB): 11.80
Params size (MB): 0.49
Estimated Total Size (MB): 27.65

In [6]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        5,760             2.96%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 96]   48,096           24.68%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 96]   [128, 60, 96]   --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─mLSTMBlock (0)                                   [128, 60, 96]   [128, 60, 96]   --                   --         True
│    │    │    │    └─LayerNorm (xlstm_norm)      

In [9]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        5,760             2.96%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 96]   48,096           24.68%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 96]   [128, 60, 96]   --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─mLSTMBlock (0)                                   [128, 60, 96]   [128, 60, 96]   61,348           31.48%         True
│    │    │    └─sLSTMBlock (1)                   

# sLSTM v mLSTM

In [93]:
task = "slstm_v_mlstm"
interval = 5
configs = ['mlstm', 'slstm', 'xlstm']
seq_size = interval * 100
context_time_mins = 10
seq_len = context_time_mins * 60 * 100 // seq_size


with open(f"./config/{task}/xlstm_mlstm_{interval}sec_config.json", "r") as file:
    config_mlstm = json.load(file)
with open(f"./config/{task}/xlstm_slstm_{interval}sec_config.json", "r") as file:
    config_slstm = json.load(file)
with open(f"./config/{task}/xlstm_xlstm_{interval}sec_config.json", "r") as file:
    config_xlstm = json.load(file)


mlstm_model = xLSTMRegressor(**config_mlstm)
slstm_model = xLSTMRegressor(**config_slstm)
xlstm_model = xLSTMRegressor(**config_xlstm)


In [94]:
summary(model=xlstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Trainable
xLSTMRegressor (xLSTMRegressor)                                   [128, 120, 500] [128, 1]        --              True
├─Linear (embedding)                                              [128, 120, 500] [128, 120, 96]  48,096          True
├─ModuleList (xlstm_layers)                                       --              --              --              True
│    └─xLSTMBlockStack (0)                                        [128, 120, 96]  [128, 120, 96]  --              True
│    │    └─ModuleList (blocks)                                   --              --              --              True
│    │    │    └─mLSTMBlock (0)                                   [128, 120, 96]  [128, 120, 96]  61,348          True
│    │    │    └─sLSTMBlock (1)                                   [128, 120, 96]  [128, 120, 96]  74,880          True
│    │    └─LayerNorm (post_blocks_norm)   

In [95]:
summary(model=mlstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Trainable
xLSTMRegressor (xLSTMRegressor)                                   [128, 120, 500] [128, 1]        --              True
├─Linear (embedding)                                              [128, 120, 500] [128, 120, 96]  48,096          True
├─ModuleList (xlstm_layers)                                       --              --              --              True
│    └─xLSTMBlockStack (0)                                        [128, 120, 96]  [128, 120, 96]  --              True
│    │    └─ModuleList (blocks)                                   --              --              --              True
│    │    │    └─mLSTMBlock (0)                                   [128, 120, 96]  [128, 120, 96]  61,348          True
│    │    └─LayerNorm (post_blocks_norm)                          [128, 120, 96]  [128, 120, 96]  96              True
├─Linear (fc)                              

In [96]:
summary(model=slstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Trainable
xLSTMRegressor (xLSTMRegressor)                                   [128, 120, 500] [128, 1]        --              True
├─Linear (embedding)                                              [128, 120, 500] [128, 120, 96]  48,096          True
├─ModuleList (xlstm_layers)                                       --              --              --              True
│    └─xLSTMBlockStack (0)                                        [128, 120, 96]  [128, 120, 96]  --              True
│    │    └─ModuleList (blocks)                                   --              --              --              True
│    │    │    └─sLSTMBlock (0)                                   [128, 120, 96]  [128, 120, 96]  74,880          True
│    │    └─LayerNorm (post_blocks_norm)                          [128, 120, 96]  [128, 120, 96]  96              True
├─Linear (fc)                              

# Comparison Baseline smaller

In [15]:
task = "comparison_baseline_cv"
interval = 5
config = "smaller"
seq_size = interval * 100
context_time_mins = 5
seq_len = context_time_mins * 60 * 100 // seq_size
with open(f"../config/{task}/lstm_{config}_{interval}sec_config.json", "r") as file:
    config_lstm = json.load(file)
with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
    config_xlstm = json.load(file)
# config = "v1"
# with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
#     config_xlstm2 = json.load(file)

lstm_model = LSTMRegressor(**config_lstm)
xlstm_model = xLSTMRegressor(**config_xlstm)
xlstm_model_v2 = xLSTMRegressor_v2(**config_xlstm)
# xlstm_model_v3 = xLSTMRegressor_v3(**config_xlstm)

# xlstm_model2 = xLSTMRegressor(**config_xlstm2)

In [16]:
summary(model=lstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                  Input Shape     Output Shape    Param #         Param %         Trainable
LSTMRegressor (LSTMRegressor)            [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                     [128, 60, 500]  [128, 60, 32]   16,032           65.40%         True
├─LSTM (lstm)                            [128, 60, 32]   [128, 60, 32]   8,448            34.46%         True
├─Linear (fc)                            [128, 32]       [128, 1]        33                0.13%         True
Total params: 24,513
Trainable params: 24,513
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 66.94
Input size (MB): 15.36
Forward/backward pass size (MB): 3.93
Params size (MB): 0.10
Estimated Total Size (MB): 19.39

In [17]:
summary(model=xlstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor (xLSTMRegressor)                                   [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 32]   16,032           60.00%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 32]   [128, 60, 32]   --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─sLSTMBlock (0)                                   [128, 60, 32]   [128, 60, 32]   --                   --         True
│    │    │    │    └─LayerNorm (xlstm_norm)      

In [18]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        1,920             6.35%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 32]   16,032           53.01%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 32]   [128, 60, 32]   --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─sLSTMBlock (0)                                   [128, 60, 32]   [128, 60, 32]   --                   --         True
│    │    │    │    └─LayerNorm (xlstm_norm)      

In [19]:
# summary(model=xlstm_model_v3, input_size=(256, seq_len, seq_size),
#         col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
#         col_width=15,
#         row_settings=["var_names"],
#         depth=7)

In [20]:
summary(model=xlstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor (xLSTMRegressor)                                   [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 32]   16,032           60.00%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 32]   [128, 60, 32]   --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─sLSTMBlock (0)                                   [128, 60, 32]   [128, 60, 32]   10,624           39.76%         True
│    │    └─LayerNorm (post_blocks_norm)          

In [21]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        1,920             6.35%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 32]   16,032           53.01%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 32]   [128, 60, 32]   --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─sLSTMBlock (0)                                   [128, 60, 32]   [128, 60, 32]   10,624           35.13%         True
│    │    └─LayerNorm (post_blocks_norm)          

In [ ]:
# summary(model=xlstm_model_v3, input_size=(256, seq_len, seq_size),
#         col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
#         col_width=15,
#         row_settings=["var_names"],
#         depth=4)

# Comparison Baseline larger

In [8]:
task = "comparison_baseline_cv"
interval = 5
config = "larger"
seq_size = interval * 100
context_time_mins = 5
seq_len = context_time_mins * 60 * 100 // seq_size
with open(f"../config/{task}/lstm_{config}_{interval}sec_config.json", "r") as file:
    config_lstm = json.load(file)
with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
    config_xlstm = json.load(file)
# config = "v1"
# with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
#     config_xlstm2 = json.load(file)

lstm_model = LSTMRegressor(**config_lstm)
xlstm_model = xLSTMRegressor(**config_xlstm)
xlstm_model_v2 = xLSTMRegressor_v2(**config_xlstm)
# xlstm_model_v3 = xLSTMRegressor_v3(**config_xlstm)

# xlstm_model2 = xLSTMRegressor(**config_xlstm2)

In [9]:
summary(model=lstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                  Input Shape     Output Shape    Param #         Param %         Trainable
LSTMRegressor (LSTMRegressor)            [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                     [128, 60, 500]  [128, 60, 192]  96,192           24.49%         True
├─LSTM (lstm)                            [128, 60, 192]  [128, 60, 192]  296,448          75.46%         True
├─Linear (fc)                            [128, 192]      [128, 1]        193               0.05%         True
Total params: 392,833
Trainable params: 392,833
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 2.29
Input size (MB): 15.36
Forward/backward pass size (MB): 23.59
Params size (MB): 1.57
Estimated Total Size (MB): 40.53

In [10]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        11,520            1.78%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 192]  96,192           14.85%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 192]  [128, 60, 192]  --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─mLSTMBlock (0)                                   [128, 60, 192]  [128, 60, 192]  233,284          36.02%         True
│    │    │    └─sLSTMBlock (1)                   

# Comparison Baseline vv-larger

In [11]:
task = "comparison_baseline_cv"
interval = 5
config = "vv_larger"
seq_size = interval * 100
context_time_mins = 5
seq_len = context_time_mins * 60 * 100 // seq_size
with open(f"../config/{task}/lstm_{config}_{interval}sec_config.json", "r") as file:
    config_lstm = json.load(file)
with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
    config_xlstm = json.load(file)
# config = "v1"
# with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
#     config_xlstm2 = json.load(file)

lstm_model = LSTMRegressor(**config_lstm)
xlstm_model = xLSTMRegressor(**config_xlstm)
xlstm_model_v2 = xLSTMRegressor_v2(**config_xlstm)
# xlstm_model_v3 = xLSTMRegressor_v3(**config_xlstm)

# xlstm_model2 = xLSTMRegressor(**config_xlstm2)

In [12]:
summary(model=lstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                  Input Shape     Output Shape    Param #         Param %         Trainable
LSTMRegressor (LSTMRegressor)            [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                     [128, 60, 500]  [128, 60, 512]  256,512          10.88%         True
├─LSTM (lstm)                            [128, 60, 512]  [128, 60, 512]  2,101,248        89.10%         True
├─Linear (fc)                            [128, 512]      [128, 1]        513               0.02%         True
Total params: 2,358,273
Trainable params: 2,358,273
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 16.17
Input size (MB): 15.36
Forward/backward pass size (MB): 62.92
Params size (MB): 9.43
Estimated Total Size (MB): 87.71

In [13]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        30,720            0.76%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 512]  256,512           6.33%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 512]  [128, 60, 512]  --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─mLSTMBlock (0)                                   [128, 60, 512]  [128, 60, 512]  --                   --         True
│    │    │    │    └─LayerNorm (xlstm_norm)      

# Comparison Baseline v-larger

In [14]:
task = "comparison_baseline_cv"
interval = 5
config = "v_larger"
seq_size = interval * 100
context_time_mins = 5
seq_len = context_time_mins * 60 * 100 // seq_size
with open(f"../config/{task}/lstm_{config}_{interval}sec_config.json", "r") as file:
    config_lstm = json.load(file)
with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
    config_xlstm = json.load(file)
# config = "v1"
# with open(f"../config/{task}/xlstm_{config}_{interval}sec_config.json", "r") as file:
#     config_xlstm2 = json.load(file)

lstm_model = LSTMRegressor(**config_lstm)
xlstm_model = xLSTMRegressor(**config_xlstm)
xlstm_model_v2 = xLSTMRegressor_v2(**config_xlstm)
# xlstm_model_v3 = xLSTMRegressor_v3(**config_xlstm)

# xlstm_model2 = xLSTMRegressor(**config_xlstm2)

In [15]:
summary(model=lstm_model, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=7)

Layer (type (var_name))                  Input Shape     Output Shape    Param #         Param %         Trainable
LSTMRegressor (LSTMRegressor)            [128, 60, 500]  [128, 1]        --                   --         True
├─Linear (embedding)                     [128, 60, 500]  [128, 60, 320]  160,320          16.32%         True
├─LSTM (lstm)                            [128, 60, 320]  [128, 60, 320]  821,760          83.65%         True
├─Linear (fc)                            [128, 320]      [128, 1]        321               0.03%         True
Total params: 982,401
Trainable params: 982,401
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 6.33
Input size (MB): 15.36
Forward/backward pass size (MB): 39.32
Params size (MB): 3.93
Estimated Total Size (MB): 58.61

In [16]:
summary(model=xlstm_model_v2, input_size=(128, seq_len, seq_size),
        col_names=["input_size", "output_size", "num_params", "params_percent", "trainable"],
        col_width=15,
        row_settings=["var_names"],
        depth=4)

Layer (type (var_name))                                           Input Shape     Output Shape    Param #         Param %         Trainable
xLSTMRegressor_v2 (xLSTMRegressor_v2)                             [128, 60, 500]  [128, 1]        19,200            1.15%         True
├─Linear (embedding)                                              [128, 60, 500]  [128, 60, 320]  160,320           9.58%         True
├─ModuleList (xlstm_layers)                                       --              --              --                   --         True
│    └─xLSTMBlockStack (0)                                        [128, 60, 320]  [128, 60, 320]  --                   --         True
│    │    └─ModuleList (blocks)                                   --              --              --                   --         True
│    │    │    └─mLSTMBlock (0)                                   [128, 60, 320]  [128, 60, 320]  634,564          37.92%         True
│    │    │    └─sLSTMBlock (1)                   